# 15 — AI Governance: Ethics, Policy & Responsible AI

**Author:** Bency (Benjamin Adjei)  
**Course:** M.S. Cybersecurity — AI Security & Governance  
**Certifications:** CompTIA AI+, CISM, CISA

---

## Objectives
- Understand AI governance frameworks (NIST AI RMF, EU AI Act, ISO 42001)
- Assess AI system risk under the EU AI Act risk tiers
- Evaluate model fairness and detect bias
- Apply explainability principles to model decisions
- Build an AI system risk register
- Generate a Responsible AI assessment report

## 1. Setup & Imports

In [ ]:
import json
import math
from datetime import datetime, timezone
from collections import Counter, defaultdict

## 2. AI Governance Landscape

### Major AI Governance Frameworks

| Framework | Issuer | Scope | Key Pillars |
|-----------|--------|-------|-------------|
| **NIST AI RMF 1.0** | NIST (US) | Voluntary | Govern, Map, Measure, Manage |
| **EU AI Act (2024)** | European Union | Mandatory (EU) | Risk-based tiers: Unacceptable → High → Limited → Minimal |
| **ISO/IEC 42001:2023** | ISO | Certifiable AIMS | AI management system standard |
| **OECD AI Principles** | OECD | Policy guidance | Inclusive growth, human-centred, transparency |
| **NIST AI 100-1** | NIST | Guidance | Trustworthy AI characteristics |
| **UK AI Safety Institute** | UK Gov | Frontier AI | Evaluation of frontier model risks |
| **IEEE 7000 Series** | IEEE | Engineering | Ethics in autonomous systems design |

### NIST AI RMF Core Functions
```
 GOVERN  ─── Establish culture, policies, accountability
   │
 MAP  ────── Identify AI risks in context
   │
 MEASURE ─── Analyse and assess identified risks
   │
 MANAGE ──── Prioritise and treat risks; respond & recover
```

## 3. EU AI Act Risk Classification

In [ ]:
EU_AI_ACT_TIERS = {
    'UNACCEPTABLE': {
        'description': 'Prohibited AI practices',
        'examples'   : [
            'Social scoring by governments',
            'Real-time biometric surveillance in public spaces (with exceptions)',
            'AI that exploits psychological vulnerabilities',
            'Emotion recognition in workplaces/education'
        ],
        'requirements': 'BANNED — cannot be deployed in the EU'
    },
    'HIGH': {
        'description': 'High-risk AI systems requiring strict compliance',
        'examples'   : [
            'AI in critical infrastructure (energy, water, transport)',
            'AI in hiring / employment decisions',
            'AI in credit scoring / loan decisions',
            'AI in medical devices',
            'AI in law enforcement (predictive policing)',
            'AI in education (student assessment)'
        ],
        'requirements': 'Conformity assessment, human oversight, transparency, accuracy, robustness'
    },
    'LIMITED': {
        'description': 'Limited risk — transparency obligations',
        'examples'   : [
            'Chatbots / conversational AI',
            'Deepfake generation systems',
            'Emotion recognition systems'
        ],
        'requirements': 'Must disclose AI interaction to users'
    },
    'MINIMAL': {
        'description': 'Minimal or no risk',
        'examples'   : [
            'AI-enabled video games',
            'Spam filters',
            'AI-powered search recommendations'
        ],
        'requirements': 'No specific obligations; follow voluntary codes of conduct'
    }
}


def classify_eu_ai_act(use_case: str) -> str:
    """Classify an AI use case under EU AI Act risk tiers."""
    HIGH_KEYWORDS     = ['hiring', 'credit', 'loan', 'medical', 'police', 'biometric',
                         'infrastructure', 'education', 'border', 'justice']
    LIMITED_KEYWORDS  = ['chatbot', 'deepfake', 'emotion', 'conversational']
    BANNED_KEYWORDS   = ['social score', 'mass surveillance', 'subliminal', 'exploit vulnerability']

    use_case_lower = use_case.lower()
    if any(k in use_case_lower for k in BANNED_KEYWORDS):  return 'UNACCEPTABLE'
    if any(k in use_case_lower for k in HIGH_KEYWORDS):    return 'HIGH'
    if any(k in use_case_lower for k in LIMITED_KEYWORDS): return 'LIMITED'
    return 'MINIMAL'


test_use_cases = [
    'AI chatbot for customer support',
    'AI model for hiring candidate screening',
    'AI credit scoring for loan approval',
    'AI spam filter for email',
    'AI system for predictive policing',
    'AI deepfake video generation tool',
    'Social score system for citizen ranking',
]

print('=== EU AI ACT RISK CLASSIFICATION ===\n')
for uc in test_use_cases:
    tier = classify_eu_ai_act(uc)
    req  = EU_AI_ACT_TIERS[tier]['requirements'][:70]
    print(f'  [{tier:<14}] {uc}')
    print(f'    Obligation: {req}')

## 4. AI Bias Detection & Fairness Metrics

In [ ]:
# Simulated loan approval model outcomes by demographic group
# Each tuple: (group, approved, total_applications)
LOAN_OUTCOMES = [
    {'group': 'Group A', 'approved': 850, 'total': 1000},
    {'group': 'Group B', 'approved': 520, 'total': 1000},
    {'group': 'Group C', 'approved': 430, 'total': 1000},
    {'group': 'Group D', 'approved': 790, 'total': 1000},
]


def compute_fairness_metrics(outcomes: list) -> dict:
    """Compute demographic parity and disparate impact across groups."""
    approval_rates = {o['group']: o['approved'] / o['total'] for o in outcomes}
    max_rate       = max(approval_rates.values())
    min_rate       = min(approval_rates.values())

    # Disparate Impact = min_approval_rate / max_approval_rate
    # EEOC 4/5ths rule: DI < 0.8 indicates potential discrimination
    disparate_impact = min_rate / max_rate

    # Demographic Parity Difference = max_rate - min_rate
    dp_difference = max_rate - min_rate

    bias_detected = disparate_impact < 0.8

    return {
        'approval_rates'   : {k: f'{v*100:.1f}%' for k, v in approval_rates.items()},
        'disparate_impact' : round(disparate_impact, 3),
        'dp_difference'    : f'{dp_difference*100:.1f}%',
        'bias_detected'    : bias_detected,
        'eeoc_threshold'   : '0.80 (4/5ths rule)',
        'verdict'          : 'BIAS DETECTED — model may violate equal opportunity laws' if bias_detected
                             else 'Acceptable fairness level'
    }


fairness = compute_fairness_metrics(LOAN_OUTCOMES)

print('=== FAIRNESS ANALYSIS: LOAN APPROVAL MODEL ===\n')
print('Approval Rates by Group:')
for group, rate in fairness['approval_rates'].items():
    bar = '█' * int(float(rate.strip('%')) // 5)
    print(f'  {group}: {rate:6}  {bar}')

print(f"\nDisparate Impact    : {fairness['disparate_impact']} (threshold: {fairness['eeoc_threshold']})")
print(f"Demographic Parity Δ: {fairness['dp_difference']}")
print(f"\nVerdict: {'⚠️  ' + fairness['verdict'] if fairness['bias_detected'] else '✅ ' + fairness['verdict']}")

## 5. Explainability — Feature Importance Analysis

In [ ]:
# Simulated SHAP-style feature importances for a credit risk model
FEATURE_IMPORTANCES = [
    {'feature': 'Credit Score',         'importance': 0.38, 'direction': '+', 'concern': False},
    {'feature': 'Debt-to-Income Ratio', 'importance': 0.22, 'direction': '-', 'concern': False},
    {'feature': 'Employment Length',    'importance': 0.14, 'direction': '+', 'concern': False},
    {'feature': 'ZIP Code',             'importance': 0.12, 'direction': '-', 'concern': True},   # Proxy for race/ethnicity
    {'feature': 'Loan Amount',          'importance': 0.08, 'direction': '-', 'concern': False},
    {'feature': 'Payment History',      'importance': 0.04, 'direction': '+', 'concern': False},
    {'feature': 'Age',                  'importance': 0.02, 'direction': '-', 'concern': True},   # Protected attribute
]

print('=== MODEL EXPLAINABILITY — FEATURE IMPORTANCE ===\n')
print(f'{"Feature":<25} {"Importance":<12} {"Direction":<12} Risk Flag')
print('-' * 65)
for f in sorted(FEATURE_IMPORTANCES, key=lambda x: x['importance'], reverse=True):
    bar      = '█' * int(f['importance'] * 50)
    flag     = '⚠️  PROXY/PROTECTED ATTRIBUTE' if f['concern'] else ''
    arrow    = '▲ Positive' if f['direction'] == '+' else '▼ Negative'
    print(f"  {f['feature']:<23} {f['importance']:<12.2f} {arrow:<12} {flag}")
    print(f"  {'':23} {bar}")

concerns = [f['feature'] for f in FEATURE_IMPORTANCES if f['concern']]
print(f"\n⚠️  Governance concern: Features {concerns} may encode protected characteristics.")
print('   Recommendation: Remove or audit these features for disparate impact.')

## 6. AI System Risk Register

In [ ]:
AI_RISK_REGISTER = [
    {
        'id'        : 'AIR-001',
        'category'  : 'Fairness & Bias',
        'risk'      : 'Loan approval model shows disparate impact on Group B/C — potential ECOA violation',
        'likelihood': 4, 'impact': 5,
        'owner'     : 'Chief AI Officer / Legal',
        'treatment' : 'Mitigate',
        'controls'  : ['Retrain with fairness constraints (equalized odds)', 'Quarterly bias audit', 'Human review for borderline decisions']
    },
    {
        'id'        : 'AIR-002',
        'category'  : 'Security',
        'risk'      : 'LLM exposed to prompt injection — could leak system prompt or take unauthorized actions',
        'likelihood': 4, 'impact': 4,
        'owner'     : 'AI Security Team',
        'treatment' : 'Mitigate',
        'controls'  : ['Deploy prompt injection detection layer', 'Isolate system prompt', 'Restrict LLM tool permissions']
    },
    {
        'id'        : 'AIR-003',
        'category'  : 'Privacy',
        'risk'      : 'Training data contains PII — model may reproduce sensitive user information',
        'likelihood': 3, 'impact': 5,
        'owner'     : 'Data Privacy Officer',
        'treatment' : 'Mitigate',
        'controls'  : ['PII scrubbing pipeline before training', 'Differential privacy during fine-tuning', 'Output PII filtering']
    },
    {
        'id'        : 'AIR-004',
        'category'  : 'Transparency',
        'risk'      : 'Model decisions not explainable — regulatory challenge under GDPR Article 22 (right to explanation)',
        'likelihood': 3, 'impact': 4,
        'owner'     : 'AI Governance Team',
        'treatment' : 'Mitigate',
        'controls'  : ['Implement SHAP/LIME explanations for all high-stakes decisions', 'Maintain model cards']
    },
    {
        'id'        : 'AIR-005',
        'category'  : 'Reliability',
        'risk'      : 'Model hallucinations in customer-facing chatbot — reputational and legal risk',
        'likelihood': 4, 'impact': 3,
        'owner'     : 'Product / AI Team',
        'treatment' : 'Mitigate',
        'controls'  : ['Retrieval-Augmented Generation (RAG) to ground responses', 'Output confidence scoring', 'Human escalation path']
    },
    {
        'id'        : 'AIR-006',
        'category'  : 'Accountability',
        'risk'      : 'No model versioning or audit trail — cannot reconstruct model decisions for regulatory review',
        'likelihood': 2, 'impact': 4,
        'owner'     : 'MLOps / Compliance',
        'treatment' : 'Mitigate',
        'controls'  : ['Implement MLflow or similar model registry', 'Log all inference requests with model version', 'Immutable audit logs']
    }
]

for risk in AI_RISK_REGISTER:
    risk['score']  = risk['likelihood'] * risk['impact']
    risk['rating'] = 'CRITICAL' if risk['score'] >= 15 else 'HIGH' if risk['score'] >= 10 else 'MEDIUM'

print('=== AI RISK REGISTER ===\n')
print(f'{"ID":<9} {"Score":<7} {"Rating":<10} {"Category":<22} Risk')
print('-' * 100)
for r in sorted(AI_RISK_REGISTER, key=lambda x: x['score'], reverse=True):
    print(f"{r['id']:<9} {r['score']:<7} {r['rating']:<10} {r['category']:<22} {r['risk'][:60]}")

## 7. NIST AI RMF Self-Assessment

In [ ]:
NIST_AI_RMF_ASSESSMENT = [
    {'function': 'GOVERN', 'subcategory': 'AI risk policies defined',             'status': 'PARTIAL'},
    {'function': 'GOVERN', 'subcategory': 'AI roles & responsibilities assigned', 'status': 'PARTIAL'},
    {'function': 'GOVERN', 'subcategory': 'Incident response plan for AI',        'status': 'NOT DONE'},
    {'function': 'MAP',    'subcategory': 'AI use case risk classification',       'status': 'DONE'},
    {'function': 'MAP',    'subcategory': 'Stakeholder impact assessment',         'status': 'PARTIAL'},
    {'function': 'MAP',    'subcategory': 'Third-party AI component inventory',    'status': 'NOT DONE'},
    {'function': 'MEASURE','subcategory': 'Bias / fairness testing',               'status': 'PARTIAL'},
    {'function': 'MEASURE','subcategory': 'Explainability evaluation',             'status': 'PARTIAL'},
    {'function': 'MEASURE','subcategory': 'Security red-teaming of AI system',     'status': 'NOT DONE'},
    {'function': 'MANAGE', 'subcategory': 'Risk treatment plans documented',       'status': 'DONE'},
    {'function': 'MANAGE', 'subcategory': 'Model monitoring in production',        'status': 'PARTIAL'},
    {'function': 'MANAGE', 'subcategory': 'Model decommission process defined',    'status': 'NOT DONE'},
]

STATUS_SCORE = {'DONE': 2, 'PARTIAL': 1, 'NOT DONE': 0}
max_score    = len(NIST_AI_RMF_ASSESSMENT) * 2
actual_score = sum(STATUS_SCORE[c['status']] for c in NIST_AI_RMF_ASSESSMENT)
pct          = actual_score / max_score * 100

print('=== NIST AI RMF SELF-ASSESSMENT ===\n')
current_fn = ''
for c in NIST_AI_RMF_ASSESSMENT:
    if c['function'] != current_fn:
        current_fn = c['function']
        print(f'\n  [{current_fn}]')
    icon = {'DONE': '✅', 'PARTIAL': '⚠️ ', 'NOT DONE': '❌'}[c['status']]
    print(f'    {icon} {c["subcategory"]:<45} {c["status"]}')

print(f'\nMaturity Score: {actual_score}/{max_score} ({pct:.0f}%)')

## 8. Responsible AI Assessment Report

In [ ]:
status_counts = Counter(c['status'] for c in NIST_AI_RMF_ASSESSMENT)
risk_counts   = Counter(r['rating'] for r in AI_RISK_REGISTER)

rai_report = {
    'report_generated' : datetime.now(timezone.utc).isoformat(),
    'system'           : 'AI-Powered Loan Approval & Customer Support Platform',
    'assessed_by'      : 'Bency (Benjamin Adjei)',
    'eu_ai_act_tier'   : 'HIGH — credit scoring is explicitly listed as high-risk',
    'nist_ai_rmf_score': f'{actual_score}/{max_score} ({pct:.0f}%)',
    'fairness': {
        'disparate_impact' : fairness['disparate_impact'],
        'bias_detected'    : fairness['bias_detected'],
        'verdict'          : fairness['verdict']
    },
    'risk_summary': {
        'total_ai_risks': len(AI_RISK_REGISTER),
        'by_rating'     : dict(risk_counts)
    },
    'nist_rmf_gaps': {
        'completed': status_counts.get('DONE', 0),
        'partial'  : status_counts.get('PARTIAL', 0),
        'not_done' : status_counts.get('NOT DONE', 0)
    },
    'top_priorities': [
        'URGENT: Retrain loan model with fairness constraints — ECOA/disparate impact exposure',
        'Implement PII scrubbing pipeline for all training data (GDPR Article 5)',
        'Deploy SHAP explainability for all high-stakes loan decisions (GDPR Article 22)',
        'Conduct AI security red-teaming before next model release',
        'Define AI incident response plan and model decommission process',
        'Complete third-party AI component inventory for supply chain risk'
    ],
    'responsible_ai_principles_status': {
        'Fairness'       : 'AT RISK — bias detected',
        'Transparency'   : 'PARTIAL — explainability not fully implemented',
        'Accountability' : 'PARTIAL — model registry in progress',
        'Privacy'        : 'AT RISK — PII in training data',
        'Safety'         : 'PARTIAL — output filtering active, red-teaming pending',
        'Reliability'    : 'PARTIAL — RAG and hallucination controls planned'
    }
}

print(json.dumps(rai_report, indent=2))

## 9. Key Takeaways

| Principle | Takeaway |
|-----------|----------|
| **Fairness** | Measure disparate impact before deployment — not after a lawsuit |
| **Transparency** | High-stakes AI decisions must be explainable (GDPR Art. 22, EU AI Act) |
| **Accountability** | Log every decision with model version — you must be able to audit it |
| **Privacy** | Training data is a liability — scrub PII, apply differential privacy |
| **Safety** | Red-team your AI like you red-team your network |
| **Human oversight** | EU AI Act mandates human oversight for all HIGH-risk AI systems |

### Responsible AI Tools
| Tool | Purpose |
|------|---------|
| **IBM OpenScale / Watson OpenScale** | AI fairness, explainability & drift monitoring |
| **Microsoft Responsible AI Toolbox** | Fairness, interpretability, error analysis |
| **Google What-If Tool** | Interactive fairness exploration |
| **Fairlearn** | Open-source fairness metrics & mitigation algorithms |
| **SHAP / LIME** | Model explainability — feature importance |
| **MLflow** | Model registry, versioning, audit trail |

---
*🎓 Series Complete — Notebooks 06–15 cover the full M.S. Cybersecurity + AI portfolio*